# Actor-Critic

In [38]:
!pip install "gymnasium[classic-control]"

   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
    --------------------------------------- 0.3/10.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.6 MB 1.9 MB/s eta 0:00:06
   --- ------------------------------------ 1.0/10.6 MB 1.9 MB/s eta 0:00:06
   ------ --------------------------------- 1.8/10.6 MB 2.4 MB/s eta 0:00:04
   -------- ------------------------------- 2.4/10.6 MB 2.9 MB/s eta 0:00:03
   -------------- ------------------------- 3.9/10.6 MB 3.5 MB/s eta 0:00:02
   ----------------- ---------------------- 4.7/10.6 MB 3.6 MB/s eta 0:00:02
   ------------------ --------------------- 5.0/10.6 MB 3.2 MB/s eta 0:00:02
   ------------------- -------------------- 5.2/10.6 MB 3.1 MB/s eta 0:00:02
   --------------------- ---------

In [35]:
!pip install imageio moviepy

  Using cached imageio-2.37.2-py3-none-any.whl.metadata (9.7 kB)
  Using cached moviepy-2.2.1-py3-none-any.whl.metadata (6.9 kB)
  Using cached imageio_ffmpeg-0.6.0-py3-none-win_amd64.whl.metadata (1.5 kB)
  Using cached proglog-0.1.12-py3-none-any.whl.metadata (794 bytes)
Using cached imageio-2.37.2-py3-none-any.whl (317 kB)
Using cached moviepy-2.2.1-py3-none-any.whl (129 kB)
Using cached proglog-0.1.12-py3-none-any.whl (6.3 kB)
Using cached imageio_ffmpeg-0.6.0-py3-none-win_amd64.whl (31.2 MB)

   ---------------------------------------- 0/4 [imageio_ffmpeg]
   ---------- ----------------------------- 1/4 [imageio]
   ------------------------------ --------- 3/4 [moviepy]
   ------------------------------ --------- 3/4 [moviepy]
   ---------------------------------------- 4/4 [moviepy]



In [39]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

In [16]:
# Actor网络：输出动作的概率分布
class PolicyNet(torch.nn.Module):
    def __init__(self, state_dim, hidden_dim, action_dim):
        super(PolicyNet, self).__init__()
        self.fc1 = torch.nn.Linear(state_dim, hidden_dim)
        self.fc2 = torch.nn.Linear(hidden_dim, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return F.softmax(self.fc2(x), dim=1) # 输出softmax转化为概率

# Critic网络：输出状态的价值
class ValueNet(torch.nn.Module):
    def __init__(self, state_dim, hidden_dim):
        super(ValueNet, self).__init__()
        self.fc1 = torch.nn.Linear(state_dim, hidden_dim)
        self.fc2 = torch.nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x) # 输出一个标量价值

In [25]:
class ActorCritic:
    def __init__(self, state_dim, hidden_dim, action_dim, actor_lr, critic_lr, gamma, device):
        # 1. 初始化网络和优化器
        self.actor = PolicyNet(state_dim, hidden_dim, action_dim).to(device)
        self.critic = ValueNet(state_dim, hidden_dim).to(device)
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=actor_lr)
        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=critic_lr)
        self.gamma = gamma # 折扣因子
        self.device = device

    def take_action(self, state):
        # 2. 根据策略选择动作，并将 state 移动到与模型相同的设备上
        state = torch.tensor([state], dtype=torch.float).to(self.device) 
        probs = self.actor(state)
        action_dist = torch.distributions.Categorical(probs)
        action = action_dist.sample()
        return action.item()

    def update(self, transition_dict):
        # 3. 从交互数据中解包，并统一移动到模型所在的设备上
        states = torch.tensor(transition_dict['states'], dtype=torch.float).to(self.device)
        actions = torch.tensor(transition_dict['actions']).view(-1, 1).to(self.device)
        rewards = torch.tensor(transition_dict['rewards'], dtype=torch.float).view(-1, 1).to(self.device)
        next_states = torch.tensor(transition_dict['next_states'], dtype=torch.float).to(self.device)
        dones = torch.tensor(transition_dict['dones'], dtype=torch.float).view(-1, 1).to(self.device)

        # 后面的 Critic 和 Actor 更新代码保持不变...

        # 4. Critic网络更新
        # 计算TD Target 和 TD Error
        td_target = rewards + self.gamma * self.critic(next_states) * (1 - dones)
        td_delta = td_target - self.critic(states) # TD误差
        
        # Critic的损失函数：TD误差的均方误差（MSE）
        critic_loss = torch.mean(F.mse_loss(self.critic(states), td_target.detach()))
        
        # 5. Actor网络更新
        # 计算动作的对数概率
        log_probs = torch.log(self.actor(states).gather(1, actions))
        # Actor的损失函数：负的（对数概率 * TD误差）
        actor_loss = torch.mean(-log_probs * td_delta.detach())

        # 6. 执行反向传播和参数更新
        self.actor_optimizer.zero_grad()
        self.critic_optimizer.zero_grad()
        actor_loss.backward()
        critic_loss.backward()
        self.actor_optimizer.step()
        self.critic_optimizer.step()

In [45]:
def train_actor_critic():
    # 设置超参数
    num_episodes = 800      # 训练的总回合数
    hidden_dim = 128        # 隐藏层维度
    gamma = 0.98            # 折扣因子
    actor_lr = 1e-3         # Actor 学习率
    critic_lr = 1e-2        # Critic 学习率 (通常比Actor大一点)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 创建环境
    env = gym.make('CartPole-v1', render_mode="rgb_array")
    env = RecordVideo(env, video_folder="./videos", episode_trigger=lambda x: (x+1) % 100 == 0)
    state_dim = env.observation_space.shape[0]  # 4维状态
    action_dim = env.action_space.n             # 2个动作 (左/右)

    # 初始化算法
    agent = ActorCritic(state_dim, hidden_dim, action_dim, actor_lr, critic_lr, gamma, device)

    # 开始训练
    for i in range(num_episodes):
        state, _ = env.reset()
        transition_dict = {
            'states': [], 'actions': [], 'next_states': [], 
            'rewards': [], 'dones': []
        }
        done = False
        episode_reward = 0

        while not done:
            action = agent.take_action(state)
            next_state, reward, done, truncated, _ = env.step(action)
            done = done or truncated  # 将正常结束和截断结束合并
            
            # 收集数据
            transition_dict['states'].append(state)
            transition_dict['actions'].append(action)
            transition_dict['next_states'].append(next_state)
            transition_dict['rewards'].append(reward)
            transition_dict['dones'].append(done)
            
            state = next_state
            episode_reward += reward

        # 一个回合结束后，使用收集到的数据进行更新
        agent.update(transition_dict)

        # 打印训练进度
        if (i + 1) % 50 == 0:
            print(f"Episode: {i+1}/{num_episodes}, Reward: {episode_reward}")

    env.close()

if __name__ == '__main__':
    train_actor_critic()

Episode: 50/800, Reward: 24.0
Episode: 100/800, Reward: 22.0
Episode: 150/800, Reward: 43.0
Episode: 200/800, Reward: 68.0
Episode: 250/800, Reward: 132.0
Episode: 300/800, Reward: 151.0
Episode: 350/800, Reward: 158.0
Episode: 400/800, Reward: 149.0
Episode: 450/800, Reward: 141.0
Episode: 500/800, Reward: 234.0
Episode: 550/800, Reward: 205.0
Episode: 600/800, Reward: 377.0
Episode: 650/800, Reward: 500.0
Episode: 700/800, Reward: 500.0
Episode: 750/800, Reward: 229.0
Episode: 800/800, Reward: 356.0


# PPO

In [76]:
class PPO:
    ''' PPO with Clip '''
    def __init__(self, state_dim, hidden_dim, action_dim, actor_lr, critic_lr, 
                 gamma=0.99, eps=0.2, lmbda=0.95, epochs=10, device='cpu'):
        self.actor = PolicyNet(state_dim, hidden_dim, action_dim).to(device)
        self.critic = ValueNet(state_dim, hidden_dim).to(device)
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=actor_lr)
        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=critic_lr)
        self.gamma = gamma
        self.eps = eps
        self.lmbda = lmbda
        self.epochs = epochs
        self.device = device

    def take_action(self, state):
        """根据当前策略选择动作，并返回动作和对应的对数概率"""
        state = torch.tensor(state, dtype=torch.float).unsqueeze(0).to(self.device)
        probs = self.actor(state)
        action_dist = torch.distributions.Categorical(probs)
        action = action_dist.sample()
        return action.item(), action_dist.log_prob(action).item()

    def update(self, transition_dict):
        """PPO 的核心更新函数"""
        states = torch.tensor(transition_dict['states'], dtype=torch.float).to(self.device)
        
        # ✅ 核心修复：去掉 .view(-1, 1)，保持为一维张量 [batch_size]
        actions = torch.tensor(transition_dict['actions'], dtype=torch.long).to(self.device)
        rewards = torch.tensor(transition_dict['rewards'], dtype=torch.float).to(self.device)
        next_states = torch.tensor(transition_dict['next_states'], dtype=torch.float).to(self.device)
        dones = torch.tensor(transition_dict['dones'], dtype=torch.float).to(self.device)
        
        # 收集数据时的旧对数概率，同样去掉 .view(-1, 1)
        old_log_probs = torch.tensor(transition_dict['log_probs'], dtype=torch.float).to(self.device)

        # 1. 计算 TD 误差 (用于计算优势值)
        # 此时 rewards, dones, critic输出都是一维的，运算完全匹配
        td_target = rewards + self.gamma * self.critic(next_states).squeeze(-1) * (1 - dones)
        td_delta = td_target - self.critic(states).squeeze(-1)
        td_delta = td_delta.detach().cpu().numpy().flatten().tolist()

        # 2. 计算 GAE
        advantage_list = []
        advantage = 0.0
        for delta in reversed(td_delta):
            advantage = self.gamma * self.lmbda * advantage + delta
            advantage_list.append(advantage)
        advantage_list.reverse()
        
        # ✅ 保持 advantage 为一维张量 [batch_size]
        advantage = torch.tensor(advantage_list, dtype=torch.float).to(self.device)
        
        # 优势值标准化
        advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-8)

        # 3. 进行多个 Epoch 的更新
        for _ in range(self.epochs):
            current_probs = self.actor(states)
            # gather 操作需要 actions 是 [batch_size, 1]，所以在这里临时升维
            current_log_probs = torch.log(current_probs.gather(1, actions.unsqueeze(-1))).squeeze(-1)
            
            # 计算重要性采样比率 (ratio)
            # 现在 old_log_probs 和 current_log_probs 都是 [batch_size]，相减绝对安全！
            ratio = torch.exp(current_log_probs - old_log_probs)
            
            # PPO 的 Clipped Surrogate Objective
            surr1 = ratio * advantage
            surr2 = torch.clamp(ratio, 1 - self.eps, 1 + self.eps) * advantage
            actor_loss = -torch.min(surr1, surr2).mean()
            
            # Critic 的均方误差 Loss
            critic_loss = F.mse_loss(self.critic(states).squeeze(-1), td_target.detach())
            
            # 梯度更新
            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()
            
            self.critic_optimizer.zero_grad()
            critic_loss.backward()
            self.critic_optimizer.step()

In [77]:
def train_PPO():
    # 设置超参数
    num_episodes = 400      # 训练的总回合数
    hidden_dim = 128        # 隐藏层维度
    gamma = 0.98            # 折扣因子
    actor_lr = 3e-4         # Actor 学习率
    critic_lr = 1e-3        # Critic 学习率 (通常比Actor大一点)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 创建环境
    env = gym.make('CartPole-v1', render_mode="rgb_array")
    env = RecordVideo(env, video_folder="./videos/PPO", episode_trigger=lambda x: (x+1) % 100 == 0)
    state_dim = env.observation_space.shape[0]  # 4维状态
    action_dim = env.action_space.n             # 2个动作 (左/右)

    agent = PPO(state_dim=state_dim, hidden_dim=hidden_dim, action_dim=action_dim, 
            actor_lr=actor_lr, critic_lr=critic_lr, gamma=gamma, device=device)

    # 开始训练
    for i in range(num_episodes):

        transition_dict = {
            'states': [], 'actions': [], 'next_states': [], 
            'rewards': [], 'dones': [], 'log_probs': []
        }
        
        # 1. 收集数据
        state, _ = env.reset()
        done = False
        episode_reward = 0  # ✅ 修复 2：初始化回合奖励，防止打印报错
        
        while not done:
            action, log_prob = agent.take_action(state)
            next_state, reward, terminated , truncated, _ = env.step(action)
            done = terminated  or truncated  # 只要正常结束或超时截断，都视为 done
            
            transition_dict['states'].append(state)
            transition_dict['actions'].append(action)
            transition_dict['next_states'].append(next_state)
            transition_dict['rewards'].append(float(reward))      # 加上 float()
            transition_dict['dones'].append(float(done))          # 加上 float()
            transition_dict['log_probs'].append(log_prob)
            
            state = next_state
            episode_reward += reward
        
        # 2. 一个 Episode 结束后，统一更新
        agent.update(transition_dict)

        # 打印训练进度
        if (i + 1) % 50 == 0:
            print(f"Episode: {i+1}/{num_episodes}, Reward: {episode_reward}")

    env.close()

if __name__ == '__main__':
    train_PPO()

Episode: 50/400, Reward: 44.0
Episode: 100/400, Reward: 35.0
Episode: 150/400, Reward: 149.0
Episode: 200/400, Reward: 219.0
Episode: 250/400, Reward: 470.0
Episode: 300/400, Reward: 500.0
Episode: 350/400, Reward: 500.0
Episode: 400/400, Reward: 500.0
